<a href="https://colab.research.google.com/github/Abhishek-S0111/Pipeline-to-Gather-Audio-Transcript-Datasets-from-Youtube/blob/main/Helper_Notebook_to_download_Audio_from_YT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Audio Transcription and Analysis

In [1]:
%%capture
!pip install yt-dlp -U
# !apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

## Audio Download and Processing

In [11]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def Download_Audio(video_ID):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': f'/content/KrishiDarshan/{video_ID}/audio.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])

In [12]:
# 1. Open an interactive input box to paste your link or ID
user_input = input("Enter YouTube Link or Video ID: ").strip()

# 2. Verify input is not empty, extract the ID, and download
if user_input:
    video_id = YT_VIDEO_ID_generator(user_input)
    print(f"Starting download for Video ID: {video_id}")
    Download_Audio(video_id)
else:
    print("Error: No link or ID was entered.")

Enter YouTube Link or Video ID: IYj3-fMzgY8
Starting download for Video ID: IYj3-fMzgY8
[youtube] Extracting URL: https://www.youtube.com/watch?v=IYj3-fMzgY8
[youtube] IYj3-fMzgY8: Downloading webpage


[youtube] IYj3-fMzgY8: Downloading android vr player API JSON
[info] IYj3-fMzgY8: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/IYj3-fMzgY8/audio.webm
[download] 100% of   21.53MiB in 00:00:02 at 8.17MiB/s   
[ExtractAudio] Destination: /content/KrishiDarshan/IYj3-fMzgY8/audio.wav
Deleting original file /content/KrishiDarshan/IYj3-fMzgY8/audio.webm (pass -k to keep)


In [14]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

# Initialize the unified Gemini client
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Test the connection using gemini-2.5-flash
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Connection successful.'
)
print(response.text)

Great! What would you like to do next?


In [34]:
import google.colab.widgets as widgets
from ipywidgets import Dropdown

# Define the available Gemini models
gemini_models = [
    'gemini-3.5-flash',
    'gemini-1.5-flash',
    'gemini-1.5-pro',
    'gemini-2.5-flash' # Added 2.5-flash
]

# Create a dropdown widget
model_selector = Dropdown(
    options=gemini_models,
    value='gemini-3.5-flash', # Default selected model
    description='Select Gemini Model:'
)

# Display the dropdown
display(model_selector)

# Assign the selected model to a variable
MODEL_NAME = model_selector.value
print(f"Selected model: {MODEL_NAME}")

# Update the MODEL_NAME when the dropdown value changes
def on_model_change(change):
    global MODEL_NAME
    MODEL_NAME = change.new
    print(f"Selected model updated to: {MODEL_NAME}")

model_selector.observe(on_model_change, names='value')

Dropdown(description='Select Gemini Model:', options=('gemini-3.5-flash', 'gemini-1.5-flash', 'gemini-1.5-pro'…

Selected model: gemini-3.5-flash


Once you have selected the desired model from the dropdown above, run the next cell to set the `MODEL_NAME` variable.

In [35]:
# This cell ensures the MODEL_NAME variable is correctly set based on the dropdown selection.
# Run this cell after making a selection in the dropdown.

# In case the notebook is run top-to-bottom without interacting with the widget,
# ensure MODEL_NAME has a default value.
if 'MODEL_NAME' not in globals():
    MODEL_NAME = 'gemini-3.5-flash' # Default if widget not interacted with
    print(f"No model selected from dropdown, defaulting to: {MODEL_NAME}")
else:
    print(f"Using selected model: {MODEL_NAME}")


Using selected model: gemini-3.5-flash


In [36]:
import time

# 1. Path to your downloaded audio file
file_path = f'/content/KrishiDarshan/{video_id}/audio.wav'

print("Uploading audio file to Gemini...")
audio_file = client.files.upload(file=file_path)

# Wait briefly for the file API to finish processing the state
print("Waiting for file processing...")
time.sleep(3)

print("\n--- Generating Transcript ---")
# 2. Generate the initial transcription
transcription_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[audio_file, "You are an expert, high-precision audio transcriber specializing in Indian agricultural media, specifically regional broadcasts like Krishi Darshan. Your task is to process the provided audio file and generate a highly detailed, chronological, verbatim transcript in hindi. Adhere strictly to the following execution rules: 1. SPEAKER IDENTIFICATION: Differentiate between speakers whenever the voice changes. If names are mentioned, use them (e.g., [Dr. Sharma]). If names are not explicitly mentioned, use logical identifiers like [Host], [Expert], or [Farmer]. 2. TIMESTAMPING RULES: Insert a precise timestamp at the beginning of EVERY speaker turn. If a single speaker talks continuously for more than 45 seconds, insert a new timestamp at the nearest natural sentence boundary. Use the exact format: [HH:MM:SS] (e.g., [00:04:12]). 3. VERBATIM FIDELITY: Transcribe the words exactly as they are spoken. Do not summarize, clean up grammar, or omit technical jargon, crop varieties, chemical names, or regional farming terms. If a technical word, brand name, or number is ambiguous or poorly pronounced, transcribe it to the best of your ability and append a [?] next to it (e.g., [Azotobacter?]). 4. OUTPUT FORMATTING: Present the final output as a clean, line-by-line log. Do not add markdown headers, introductory text, or conversational explanations. Ensure each entry follows this exact template: [TIMESTAMP] [SPEAKER]: Spoken text goes here. Example Output Line: [00:01:15] [Host]: Namaskar kisan bhaiyon, aaj hum baat karenge kharif ki fasal ke baare mein. [00:01:30] [Expert]: Kisan bhaiyon, is samay naye beejon ka chayan karna bahut mahatvapurna hai.."]
)
print(transcription_response.text)

print("\n--- Starting Conversation Mode ---")
# 3. Create an ongoing chat session with the audio context
chat = client.chats.create(model=MODEL_NAME)

# FIXED: Changed 'contents=' to 'message=' to align with the new SDK
chat.send_message(message=[audio_file, "Analyze this audio file. I will now ask you questions about its contents."])

print("Chat initialized! You can now ask questions about the audio below. (Type 'exit' to stop)")

# 4. Interactive loop to converse with the audio
while True:
    user_input = input("\nYour Question: ")
    if user_input.lower() == 'exit':
        print("Ending chat session.")
        break

    if user_input.strip() == '':
        continue

    response = chat.send_message(user_input)
    print(f"\nGemini: {response.text}")

Uploading audio file to Gemini...
Waiting for file processing...

--- Generating Transcript ---


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 26.096225538s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'},

In [24]:
# The model names in the existing cell need to be updated to use the MODEL_NAME variable.
# Please manually update the model name in the cell below to `MODEL_NAME`.
# For example, change `model='gemini-3.5-flash'` to `model=MODEL_NAME`
# and `client.chats.create(model='gemini-3.5-flash')` to `client.chats.create(model=MODEL_NAME)`

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive if it isn't already mounted
try:
    drive.mount('/content/drive')
except Exception:
    pass

# 2. Your precise destination base directory path
base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript"

# 3. Automatically inherits the video_id variable from the previous download cell
if 'video_id' not in locals() and 'video_id' not in globals():
    raise NameError("Variable 'video_id' not found. Please make sure to run your interactive download cell first!")

# 4. Dynamically append the video ID folder to your base path
target_folder = os.path.join(base_path, video_id)

# 5. Automatically create the directory if it doesn't exist yet
os.makedirs(target_folder, exist_ok=True)

# 6. Define the full text file destination
transcript_path = os.path.join(target_folder, "transcript.txt")

# 7. Write a new file or completely overwrite/update the old file with the latest text
with open(transcript_path, "w", encoding="utf-8") as f:
    f.write(transcription_response.text)

print(f"✨ Success! transcript.txt file added: {transcript_path}")